## Importing necessary libraries

In [1]:


import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import country_converter as coco

import seaborn as sns
import missingno as msno
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import iplot
from PIL import Image
import requests

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

import warnings
warnings.filterwarnings('ignore')

In [2]:
books = pd.read_csv("../data/btech books.csv")
ratings = pd.read_csv("../data/btech ratings.csv")
users = pd.read_csv("../data/btech users.csv")

In [3]:
books.columns

Index(['Branch', 'Book Title', 'Year of Publication', 'Author', 'Semester',
       'Price'],
      dtype='object')

In [4]:
ratings.columns

Index(['UserID', 'Semester', 'Ratings'], dtype='object')

In [5]:
users.columns

Index(['UserID', 'Semester'], dtype='object')

In [6]:
books.columns
ratings.columns
users.columns

Index(['UserID', 'Semester'], dtype='object')

## SQL connection

In [7]:
import mysql.connector
from mysql.connector import Error

# Database connection details
HOST = "localhost"
USER = "root"
PASSWORD = "root"

try:
    # Connect to MySQL server
    connection = mysql.connector.connect(
        host=HOST,
        user=USER,
        password=PASSWORD
    )

    if connection.is_connected():
        print("✅ Connected to MySQL Server")

        # Create a cursor object
        cursor = connection.cursor()

        # Step 1: Create the database
        cursor.execute("CREATE DATABASE IF NOT EXISTS educationbr")
        print("✅ Database 'educationbookrecommendation' created successfully")

        # Switch to the new database
        cursor.execute("USE educationbr")

        # Step 2: Create Books table
        create_books_table = """
        CREATE TABLE IF NOT EXISTS Books (
           
            Branch VARCHAR(100) NOT NULL,
            Book_Title VARCHAR(255) NOT NULL,
            Year_Of_Publication INT NOT NULL,
            Author VARCHAR(255) NOT NULL,
            Semester INT NOT NULL,
            Price DECIMAL(10,2)
        )
        """
        cursor.execute(create_books_table)
        print("✅ Table 'Books' created successfully")

        # Step 3: Create Users table
        create_users_table = """
        CREATE TABLE IF NOT EXISTS Users (
            User_ID INT PRIMARY KEY,
             Semester INT NOT NULL
        )
        """
        cursor.execute(create_users_table)
        print("✅ Table 'Users' created successfully")

        # Step 4: Create Ratings table
        create_ratings_table = """
        CREATE TABLE IF NOT EXISTS Ratings (
            User_ID INT NOT NULL,
            Semester INT NOT NULL,
            Ratings INT CHECK (Ratings BETWEEN 1 AND 10),
            PRIMARY KEY (User_ID, Semester),
            FOREIGN KEY (User_ID) REFERENCES Users(User_ID) ON DELETE CASCADE
        )
        """
        cursor.execute(create_ratings_table)
        print("✅ Table 'Ratings' created successfully")

      

except Error as e:
    print(f"❌ Error: {e}")

finally:
    # Close the connection
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("✅ MySQL connection is closed")


✅ Connected to MySQL Server
✅ Database 'educationbookrecommendation' created successfully
✅ Table 'Books' created successfully
✅ Table 'Users' created successfully
✅ Table 'Ratings' created successfully
✅ MySQL connection is closed


## Inserting the data

In [8]:
import mysql.connector
from mysql.connector import Error
import pandas as pd

# Database connection details
HOST = "localhost"
USER = "root"
PASSWORD = "root"
DATABASE = "educationbr"

# File paths
BOOKS_CSV = "../data/btech books.csv"
USERS_CSV = "../data/btech users.csv"
RATINGS_CSV = "../data/btech ratings.csv"

try:
    # Connect to MySQL
    connection = mysql.connector.connect(
        host=HOST,
        user=USER,
        password=PASSWORD,
        database=DATABASE
    )

    if connection.is_connected():
        print("✅ Connected to MySQL Database")
        cursor = connection.cursor()

        ### 🔹 Insert data into Users table ###
        users_df = pd.read_csv(USERS_CSV)

        users_df.rename(columns={"UserID": "User_ID"}, inplace=True)

        for _, row in users_df.iterrows():
            try:
                cursor.execute("""
                    INSERT IGNORE INTO Users (User_ID, Semester)
                    VALUES (%s, %s)
                """, (int(row["User_ID"]), int(row["Semester"])))  
            except Exception as e:
                print(f"❌ Error inserting user {row['User_ID']}: {e}")

        connection.commit()  # ✅ Commit after Users table
        print("✅ Users data inserted successfully")

        ### 🔹 Insert data into Books table ###
        books_df = pd.read_csv(BOOKS_CSV)

        books_df.rename(columns={
            "Book Title": "Book_Title",
            "Year of Publication": "Year_Of_Publication"
        }, inplace=True)

        for _, row in books_df.iterrows():
            try:
                cursor.execute("""
                    INSERT IGNORE INTO Books (Branch, Book_Title, Year_Of_Publication, Author, Semester, Price)
                    VALUES (%s, %s, %s, %s, %s, %s)
                """, (row["Branch"], row["Book_Title"], int(row["Year_Of_Publication"]), 
                      row["Author"], int(row["Semester"]), 
                      float(row["Price"]) if pd.notna(row["Price"]) else None))
            except Exception as e:
                print(f"❌ Error inserting book {row['Book_Title']}: {e}")

        connection.commit()  # ✅ Commit after Books table
        print("✅ Books data inserted successfully")

        ### 🔹 Insert data into Ratings table ###
        ratings_df = pd.read_csv(RATINGS_CSV)

        ratings_df.rename(columns={"UserID": "User_ID"}, inplace=True)

        for _, row in ratings_df.iterrows():
            try:
                cursor.execute("""
                    INSERT IGNORE INTO Ratings (User_ID, Semester, Ratings)
                    VALUES (%s, %s, %s)
                """, (int(row["User_ID"]), int(row["Semester"]), 
                      int(row["Ratings"]) if pd.notna(row["Ratings"]) else None))
            except mysql.connector.IntegrityError as e:
                print(f"⚠️ Duplicate entry skipped for User_ID: {row['User_ID']} in Semester: {row['Semester']}")
            except Exception as e:
                print(f"❌ Error inserting rating {row['User_ID']}: {e}")

        connection.commit()  # ✅ Commit after Ratings table
        print("✅ Ratings data inserted successfully")

except Error as e:
    print(f"❌ MySQL Error: {e}")

finally:
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("✅ MySQL connection is closed")


✅ Connected to MySQL Database
✅ Users data inserted successfully
✅ Books data inserted successfully
✅ Ratings data inserted successfully
✅ MySQL connection is closed


## Extracting the data

In [9]:
import mysql.connector
import pandas as pd

# ✅ Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="educationbr"
)
cursor = conn.cursor()

# ✅ Define function to fetch data from a table
def fetch_table_data(table_name):
    query = f"SELECT * FROM {table_name};"
    cursor.execute(query)
    rows = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    return pd.DataFrame(rows, columns=columns)

# ✅ Fetch data from each table
users_df = fetch_table_data("users")
books_df = fetch_table_data("books")
ratings_df = fetch_table_data("ratings")

# ✅ Close the connection
cursor.close()
conn.close()

# ✅ Display the first 5 rows of each table
print("Users Table:")
display(users_df.head(), "\n")

print("Books Table:")
display(books_df.head(), "\n")

print("Ratings Table:")
display(ratings_df.head(), "\n")


Users Table:


,User_ID,Semester
0,1,3
1,2,6
2,3,1
3,4,1
4,5,1


'\n'

Books Table:


,Branch,Book_Title,Year_Of_Publication,Author,Semester,Price
0,Chemical Engineering,Heat Transfer - A Comprehensive Guide,2010,Stuart Russell,6,2127.00
1,Computer Science Engineering,Computer Networks - A Comprehensive Guide,2023,Robert L. Norton,6,1611.00
2,Computer Science Engineering,Operating Systems - A Comprehensive Guide,2006,K.S. Mano,6,1826.00
3,Electronics and Communication Engineering,Digital Electronics - A Comprehensive Guide,2010,Robert L. Norton,4,1002.00
4,Mechanical Engineering,Machine Design - A Comprehensive Guide,2016,J. B. Gupta,2,527.00


'\n'

Ratings Table:


,User_ID,Semester,Ratings
0,1,2,3
1,1,3,2
2,2,1,4
3,2,7,1
4,2,8,1


'\n'

# Filtering

## Content based filtering

In [1]:
import panel as pn
import pandas as pd
import mysql.connector
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pn.extension()

# ✅ Connect to MySQL
def get_db_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="root",
        database="educationbr"
    )

# ✅ Fetch books dataset
def get_books_data():
    conn = get_db_connection()
    query = "SELECT * FROM books;"
    books_df = pd.read_sql(query, conn)
    conn.close()
    return books_df

# ✅ Fetch existing users
def get_users():
    conn = get_db_connection()
    query = "SELECT DISTINCT User_ID FROM ratings;"
    users_df = pd.read_sql(query, conn)
    conn.close()
    return users_df['User_ID'].tolist()

# ✅ Fetch distinct branches from books dataset
def get_branches():
    conn = get_db_connection()
    query = "SELECT DISTINCT Branch FROM books;"
    branches_df = pd.read_sql(query, conn)
    conn.close()
    return branches_df['Branch'].tolist()

# ✅ Content-Based Recommendation Function
def recommend_books(branch, semester):
    books_df = get_books_data()

    # Filter books based on branch and semester
    books_df = books_df[(books_df["Branch"] == branch) & (books_df["Semester"] == semester)]

    # Combine features for similarity
    books_df["combined_features"] = books_df["Branch"] + " " + books_df["Book_Title"] + " " + books_df["Author"]
    
    # ✅ TF-IDF Vectorization
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(books_df["combined_features"])

    # ✅ Compute similarity
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    # Recommend top books
    book_indices = cosine_sim.sum(axis=0).argsort()[-5:][::-1]  # Top 5 books
    recommended_books = books_df.iloc[book_indices][["Book_Title", "Author", "Branch", "Semester"]]

    return recommended_books

# ✅ Panel UI Components
users = get_users()
branches = get_branches()

user_select = pn.widgets.Select(name="Select User", options=["New User"] + users)
branch_select = pn.widgets.Select(name="Select Branch", options=branches)

new_user_id = pn.widgets.IntInput(name="New User ID", placeholder="Enter User ID")
new_user_semester = pn.widgets.IntInput(name="New User Semester", placeholder="Enter Semester")

# ✅ Show all semesters for existing users (1 to 8)
all_semesters = [str(i) for i in range(1, 9)]
semester_select = pn.widgets.Select(name="Select Semester", options=all_semesters, visible=False)

recommend_button = pn.widgets.Button(name="Get Recommendations", button_type="primary")
output = pn.pane.DataFrame()

# ✅ Function to Update UI
def update_ui(event):
    if user_select.value == "New User":
        new_user_id.visible = True
        new_user_semester.visible = True
        semester_select.visible = False  # Hide semester dropdown for new users
    else:
        new_user_id.visible = False
        new_user_semester.visible = False
        semester_select.visible = True  # Show semester dropdown for existing users

user_select.param.watch(update_ui, "value")

# ✅ Function to Display Recommendations
def get_recommendations(event):
    branch = branch_select.value
    
    if user_select.value == "New User":
        semester = new_user_semester.value
    else:
        semester = semester_select.value

    recommended_books = recommend_books(branch, int(semester))
    output.object = recommended_books

recommend_button.on_click(get_recommendations)

# ✅ Panel Layout
layout = pn.Column(
    pn.pane.Markdown("## 📚 Book Recommendation System"),
    user_select,
    new_user_id,
    new_user_semester,
    branch_select,
    semester_select,
    recommend_button,
    output
)

layout.servable()


C:\Users\SREE GANESHA\AppData\Local\Temp\ipykernel_14112\3584099726.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  users_df = pd.read_sql(query, conn)
C:\Users\SREE GANESHA\AppData\Local\Temp\ipykernel_14112\3584099726.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  branches_df = pd.read_sql(query, conn)


Column
    [0] Markdown(str)
    [1] Select(options=['New User', 1, ...], value='New User')
    [2] IntInput(name='New User ID', placeholder='Enter User ID')
    [3] IntInput(name='New User Semester', placeholder='Enter Semester')
    [4] Select(options=['Chemical Engineering', ...], value='Chemical Engineering')
    [5] Select(options=['1', '2', '3', ...], value='1', visible=False)
    [6] Button(button_type='primary', name='Get Recommendations')
    [7] DataFrame(None)

#

In [18]:
import panel as pn
import pandas as pd
import mysql.connector
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pn.extension()

# ✅ Connect to MySQL
def get_db_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="root",
        database="educationbr"
    )

# ✅ Fetch books dataset
def get_books_data():
    conn = get_db_connection()
    query = "SELECT * FROM books;"
    books_df = pd.read_sql(query, conn)
    conn.close()
    return books_df

# ✅ Fetch existing users
def get_users():
    conn = get_db_connection()
    query = "SELECT DISTINCT User_ID FROM ratings;"
    users_df = pd.read_sql(query, conn)
    conn.close()
    return users_df['User_ID'].tolist()

# ✅ Fetch distinct branches from books dataset
def get_branches():
    conn = get_db_connection()
    query = "SELECT DISTINCT Branch FROM books;"
    branches_df = pd.read_sql(query, conn)
    conn.close()
    return branches_df['Branch'].tolist()

# ✅ Content-Based Recommendation Function
def recommend_books(branch, semester):
    books_df = get_books_data()

    # Filter books based on branch and semester
    books_df = books_df[(books_df["Branch"] == branch) & (books_df["Semester"] == semester)]

    # Combine features for similarity
    books_df["combined_features"] = books_df["Branch"] + " " + books_df["Book_Title"] + " " + books_df["Author"]
    
    # ✅ TF-IDF Vectorization
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(books_df["combined_features"])

    # ✅ Compute similarity
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    # Recommend top distinct books
    books_df["similarity_score"] = cosine_sim.sum(axis=0)
    recommended_books = books_df.sort_values(by="similarity_score", ascending=False).drop_duplicates(subset=["Book_Title"]).head(5)
    
    return recommended_books[["Book_Title", "Author", "Branch", "Semester"]]

# ✅ Panel UI Components
users = get_users()
branches = get_branches()

user_select = pn.widgets.Select(name="Select User", options=["New User"] + users)
branch_select = pn.widgets.Select(name="Select Branch", options=branches)

new_user_id = pn.widgets.IntInput(name="New User ID", placeholder="Enter User ID")
new_user_semester = pn.widgets.IntInput(name="New User Semester", placeholder="Enter Semester")

# ✅ Show all semesters for existing users (1 to 8)
all_semesters = [str(i) for i in range(1, 9)]
semester_select = pn.widgets.Select(name="Select Semester", options=all_semesters, visible=False)

recommend_button = pn.widgets.Button(name="Get Recommendations", button_type="primary")
output = pn.pane.DataFrame()

# ✅ Function to Update UI
def update_ui(event):
    if user_select.value == "New User":
        new_user_id.visible = True
        new_user_semester.visible = True
        semester_select.visible = False  # Hide semester dropdown for new users
    else:
        new_user_id.visible = False
        new_user_semester.visible = False
        semester_select.visible = True  # Show semester dropdown for existing users

user_select.param.watch(update_ui, "value")

# ✅ Function to Display Recommendations
def get_recommendations(event):
    branch = branch_select.value
    
    if user_select.value == "New User":
        semester = new_user_semester.value
    else:
        semester = semester_select.value

    recommended_books = recommend_books(branch, int(semester))
    output.object = recommended_books

recommend_button.on_click(get_recommendations)

# ✅ Panel Layout
layout = pn.Column(
    pn.pane.Markdown("## 📚 Book Recommendation System"),
    user_select,
    new_user_id,
    new_user_semester,
    branch_select,
    semester_select,
    recommend_button,
    output
)

layout.servable()

Column
    [0] Markdown(str)
    [1] Select(options=['New User', 1, ...], value='New User')
    [2] IntInput(name='New User ID', placeholder='Enter User ID')
    [3] IntInput(name='New User Semester', placeholder='Enter Semester')
    [4] Select(options=['Chemical Engineering', ...], value='Chemical Engineering')
    [5] Select(options=['1', '2', '3', ...], value='1', visible=False)
    [6] Button(button_type='primary', name='Get Recommendations')
    [7] DataFrame(None)

## Colloborative filtering

In [20]:
import panel as pn
import pandas as pd
import mysql.connector

# Connect to MySQL database
def fetch_data(query, params=None):
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="root",
        database="educationbr"
    )
    cursor = conn.cursor(dictionary=True)
    cursor.execute(query, params)
    data = cursor.fetchall()
    cursor.close()
    conn.close()
    return pd.DataFrame(data)

# Fetch User IDs and Branches
user_df = fetch_data("SELECT DISTINCT User_ID FROM Ratings")
branches_df = fetch_data("SELECT DISTINCT Branch FROM Books")

user_ids = sorted(user_df["User_ID"].astype(str).tolist()) + ["New User"]
branches = sorted(branches_df["Branch"].astype(str).tolist())

# UI Components
user_select = pn.widgets.Select(name='User ID', options=user_ids)
new_user_input = pn.widgets.TextInput(name='Enter New User ID', placeholder='Enter User ID', visible=False)
semester_select = pn.widgets.Select(name='Semester', options=[str(i) for i in range(1, 9)], visible=False)
new_semester_input = pn.widgets.Select(name='Enter New Semester', options=[str(i) for i in range(1, 9)], visible=False)
branch_select = pn.widgets.Select(name='Branch', options=branches, visible=False)
recommend_button = pn.widgets.Button(name='Get Recommendations', button_type='primary')

# Function to update UI on user selection
def update_ui(event):
    if user_select.value == "New User":
        new_user_input.visible = True
        new_semester_input.visible = True
        semester_select.visible = False
    else:
        new_user_input.visible = False
        new_semester_input.visible = False
        semester_select.visible = True
    branch_select.visible = True

user_select.param.watch(update_ui, 'value')

# Fetch recommended books
def get_recommendations():
    selected_user = user_select.value
    selected_branch = branch_select.value
    
    if selected_user == "New User":
        selected_semester = int(new_semester_input.value)
    else:
        selected_semester = int(semester_select.value)
    
    next_semesters = [selected_semester + i for i in range(1, 3) if selected_semester + i <= 8]
    if not next_semesters:
        return "No further semester recommendations available."
    
    query = """
        SELECT DISTINCT Book_Title, Author, Year_of_Publication, Price, Semester
        FROM Books
        WHERE Semester IN (%s)
        AND Branch = %s
        ORDER BY Price ASC
        LIMIT 5
    """ % (', '.join(map(str, next_semesters)), '%s')
    
    books_df = fetch_data(query, (selected_branch,))
    if books_df.empty:
        return "No books found for the selected criteria."
    
    return books_df.drop_duplicates(subset=['Book_Title'])

# Callback for button
def show_recommendations(event):
    recommendations.object = get_recommendations()

recommend_button.on_click(show_recommendations)
recommendations = pn.pane.DataFrame()

# Layout
dashboard = pn.Column(
    "## Book Recommendation System",
    user_select,
    new_user_input,
    semester_select,
    new_semester_input,
    branch_select,
    recommend_button,
    recommendations
)

dashboard.servable()


Column
    [0] Markdown(str)
    [1] Select(name='User ID', options=['1', '10', '100', ...], value='1')
    [2] TextInput(name='Enter New User ID', placeholder='Enter User ID', visible=False)
    [3] Select(name='Semester', options=['1', '2', '3', ...], value='1', visible=False)
    [4] Select(name='Enter New Semester', options=['1', '2', '3', ...], value='1', visible=False)
    [5] Select(name='Branch', options=['Aerospace Engineering', ...], value='Aerospace Engineering', visible=False)
    [6] Button(button_type='primary', name='Get Recommendations')
    [7] DataFrame(None)

## Hybrid filtering

In [29]:
import panel as pn
import pandas as pd
import mysql.connector

# Connect to MySQL database
def fetch_data(query, params=None):
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="root",
        database="educationbr"
    )
    cursor = conn.cursor(dictionary=True)
    cursor.execute(query, params)
    data = cursor.fetchall()
    cursor.close()
    conn.close()
    return pd.DataFrame(data)

# Fetch User IDs and Branches
user_df = fetch_data("SELECT DISTINCT User_ID FROM Ratings")
branches_df = fetch_data("SELECT DISTINCT Branch FROM Books")

user_ids = sorted(user_df["User_ID"].astype(str).tolist()) + ["New User"]
branches = sorted(branches_df["Branch"].astype(str).tolist())

# UI Components
user_select = pn.widgets.Select(name='User ID', options=user_ids)
new_user_input = pn.widgets.TextInput(name='Enter New User ID', placeholder='Enter User ID', visible=False)
semester_select = pn.widgets.Select(name='Semester', options=[str(i) for i in range(1, 9)], visible=False)
new_semester_input = pn.widgets.Select(name='Enter New Semester', options=[str(i) for i in range(1, 9)], visible=False)
branch_select = pn.widgets.Select(name='Branch', options=branches, visible=False)
recommend_button = pn.widgets.Button(name='Get Recommendations', button_type='primary')

# Function to update UI on user selection
def update_ui(event):
    if user_select.value == "New User":
        new_user_input.visible = True
        new_semester_input.visible = True
        semester_select.visible = False
    else:
        new_user_input.visible = False
        new_semester_input.visible = False
        semester_select.visible = True
    branch_select.visible = True

user_select.param.watch(update_ui, 'value')

# Fetch recommended books
def get_recommendations():
    selected_user = user_select.value
    selected_branch = branch_select.value
    
    if selected_user == "New User":
        selected_semester = int(new_semester_input.value)
    else:
        selected_semester = int(semester_select.value)

    # Content-Based: Get books from the selected semester
    query_content = """
        SELECT DISTINCT Book_Title, Author, Semester, Branch
        FROM Books
        WHERE Semester = %s
        AND Branch = %s
    """
    content_books = fetch_data(query_content, (selected_semester, selected_branch))

    # Collaborative Filtering: Get books from next 2 consecutive semesters
    next_semesters = [selected_semester, selected_semester + 1, selected_semester + 2]
    next_semesters = [s for s in next_semesters if s <= 8]  # Ensure semester is within range

    query_collab = """
        SELECT DISTINCT Book_Title, Author, Semester, Branch
        FROM Books
        WHERE Semester IN (%s)
        AND Branch = %s
    """ % (', '.join(map(str, next_semesters)), '%s')

    collab_books = fetch_data(query_collab, (selected_branch,))

    # Combine both recommendations
    hybrid_books = pd.concat([content_books, collab_books]).drop_duplicates().sample(n=min(8, len(content_books) + len(collab_books)))

    if hybrid_books.empty:
        return "No books found for the selected criteria."
    
    return hybrid_books

# Callback for button
def show_recommendations(event):
    recommendations.object = get_recommendations()

recommend_button.on_click(show_recommendations)
recommendations = pn.pane.DataFrame()

# Layout
dashboard = pn.Column(
    "## Book Recommendation System",
    user_select,
    new_user_input,
    semester_select,
    new_semester_input,
    branch_select,
    recommend_button,
    recommendations
)

dashboard.servable()


Column
    [0] Markdown(str)
    [1] Select(name='User ID', options=['1', '10', '100', ...], value='1')
    [2] TextInput(name='Enter New User ID', placeholder='Enter User ID', visible=False)
    [3] Select(name='Semester', options=['1', '2', '3', ...], value='1', visible=False)
    [4] Select(name='Enter New Semester', options=['1', '2', '3', ...], value='1', visible=False)
    [5] Select(name='Branch', options=['Aerospace Engineering', ...], value='Aerospace Engineering', visible=False)
    [6] Button(button_type='primary', name='Get Recommendations')
    [7] DataFrame(None)

In [32]:
import panel as pn
import pandas as pd
import mysql.connector

# Connect to MySQL database
def fetch_data(query, params=None):
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="root",
        database="educationbr"
    )
    cursor = conn.cursor(dictionary=True)
    cursor.execute(query, params)
    data = cursor.fetchall()
    cursor.close()
    conn.close()
    return pd.DataFrame(data)

# Fetch User IDs and Branches
user_df = fetch_data("SELECT DISTINCT User_ID FROM Ratings")
branches_df = fetch_data("SELECT DISTINCT Branch FROM Books")

user_ids = sorted(user_df["User_ID"].astype(str).tolist()) + ["New User"]
branches = sorted(branches_df["Branch"].astype(str).tolist())

# UI Components
user_select = pn.widgets.Select(name='User ID', options=user_ids)
new_user_input = pn.widgets.TextInput(name='Enter New User ID', placeholder='Enter User ID', visible=False)
semester_select = pn.widgets.Select(name='Semester', options=[str(i) for i in range(1, 9)], visible=False)
new_semester_input = pn.widgets.Select(name='Enter New Semester', options=[str(i) for i in range(1, 9)], visible=False)
branch_select = pn.widgets.Select(name='Branch', options=branches, visible=False)
recommend_button = pn.widgets.Button(name='Get Recommendations', button_type='primary')

# Function to update UI on user selection
def update_ui(event):
    if user_select.value == "New User":
        new_user_input.visible = True
        new_semester_input.visible = True
        semester_select.visible = False
    else:
        new_user_input.visible = False
        new_semester_input.visible = False
        semester_select.visible = True
    branch_select.visible = True

user_select.param.watch(update_ui, 'value')

# Fetch recommended books
def get_recommendations():
    selected_user = user_select.value
    selected_branch = branch_select.value
    
    if selected_user == "New User":
        selected_semester = int(new_semester_input.value)
    else:
        selected_semester = int(semester_select.value)

    # Get the next two consecutive semesters
    next_semesters = [selected_semester + i for i in range(0, 3) if selected_semester + i <= 8]

    # Fetch books from these semesters
    query = """
        SELECT DISTINCT Book_Title, Author, Semester, Branch
        FROM Books
        WHERE Semester IN (%s)
        AND Branch = %%s
    """ % (', '.join(map(str, next_semesters)))

    books_df = fetch_data(query, (selected_branch,))

    # Remove duplicate book titles (keeping the first occurrence)
    books_df = books_df.drop_duplicates(subset=['Book_Title'])

    if books_df.empty:
        return "No books found for the selected criteria."
    
    return books_df

# Callback for button
def show_recommendations(event):
    recommendations.object = get_recommendations()

recommend_button.on_click(show_recommendations)
recommendations = pn.pane.DataFrame()

# Layout
dashboard = pn.Column(
    "## Book Recommendation System",
    user_select,
    new_user_input,
    semester_select,
    new_semester_input,
    branch_select,
    recommend_button,
    recommendations
)

dashboard.servable()


Column
    [0] Markdown(str)
    [1] Select(name='User ID', options=['1', '10', '100', ...], value='1')
    [2] TextInput(name='Enter New User ID', placeholder='Enter User ID', visible=False)
    [3] Select(name='Semester', options=['1', '2', '3', ...], value='1', visible=False)
    [4] Select(name='Enter New Semester', options=['1', '2', '3', ...], value='1', visible=False)
    [5] Select(name='Branch', options=['Aerospace Engineering', ...], value='Aerospace Engineering', visible=False)
    [6] Button(button_type='primary', name='Get Recommendations')
    [7] DataFrame(None)

## Saving models